# Build Task Bank

Downloads a HuggingFace dataset and converts it into task objects defined in `models/task.py`.  
Currently configured for **Summarization** using [EdinburghNLP/xsum](https://huggingface.co/datasets/EdinburghNLP/xsum).

**XSum columns**
| Column | Type | Description |
|--------|------|-------------|
| `document` | string | Full news article (model input) |
| `summary` | string | One-sentence extreme summary (ground truth) |
| `id` | string | BBC article ID |

**Splits:** train (204 045) · validation (11 332) · test (11 334)

## 1. Imports & config

In [ ]:
import json
import random
import sys
from datetime import date
from pathlib import Path
from typing import Literal

import nltk
import textstat
from datasets import load_dataset

# Make models/ importable from data_pipeline/
sys.path.append(str(Path("__file__").resolve().parent.parent))
from models.task import SummarizationTask, Task

# ── Config ────────────────────────────────────────────────────────────────────
DATASET_NAME   = "EdinburghNLP/xsum"
OUTPUT_FILE    = Path("../task_bank/summarization_tasks.jsonl")
SEED           = 42

# How many tasks to sample per split (set to None to use all)
N_TRAIN        = 500
N_VALIDATION   = 100
N_TEST         = 100

random.seed(SEED)

In [ ]:
# Download required nltk data (runs fast if already cached)
nltk.download("punkt",        quiet=True)
nltk.download("punkt_tab",    quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

## 2. Load dataset

In [ ]:
dataset = load_dataset(DATASET_NAME)
print(dataset)

In [9]:
# Sanity check — inspect one example
example = dataset["train"][0]
print("ID       :", example["id"])
print("Summary  :", example["summary"])
print("Document :", example["document"][:300], "...")

ID       : 35232142
Summary  : Clean-up operations are continuing across the Scottish Borders and Dumfries and Galloway after flooding caused by Storm Frank.
Document : The full cost of damage in Newton Stewart, one of the areas worst affected, is still being assessed.
Repair work is ongoing in Hawick and many roads in Peeblesshire remain badly affected by standing water.
Trains on the west coast mainline face disruption due to damage at the Lamington Viaduct.
Many ...


## 3. Helper functions

In [13]:
import re

# POS tags that count as content words for lexical density
_CONTENT_TAGS = {
    "NN", "NNS", "NNP", "NNPS",
    "VB", "VBD", "VBG", "VBN", "VBP", "VBZ",
    "JJ", "JJR", "JJS",
    "RB", "RBR", "RBS",
}

_DOMAIN_KEYWORDS: dict[str, dict[str, list[str]]] = {
    "politics": {
        "primary": [
            "parliament", "government", "minister", "election", "vote",
            "policy", "senate", "congress", "cabinet", "legislation",
            "political", "president", "chancellor", "referendum", "debate",
            "constituency", "campaign", "ballot", "opposition", "diplomat",
            "treaty", "sanction", "summit", "foreign secretary", "prime minister",
        ],
        "secondary": [
            r"\bmp\b", r"\bmps\b", "tory", "labour", "conservative", "liberal",
            "democrat", "republican", "shadow cabinet", "manifesto",
            "by-election", "party leader", "whitehall", "downing street",
            "westminster", "general election", "coalition", "brussels",
            "nato", "united nations", r"\bun\b", "european union",
        ],
    },
    "crime": {
        "primary": [
            "police", "court", "guilty", "sentenced", "arrested", "charged",
            "convicted", "verdict", "trial", "murder", "suspect", "victim",
            "prosecution", "defendant", "detective", "investigation",
            "stabbing", "shooting", "kidnap", "smuggling",
        ],
        "secondary": [
            "prison", "jail", "manslaughter", "assault", "robbery", "theft",
            "fraud", "burglary", "arson", "homicide", "crown court",
            "magistrate", "plea", "inquest", "parole", "probation",
            "constabulary", "warrant", "custody", "sentence", "acquitted",
            "cctv", "witness", "forensic", "bail",
        ],
    },
    "sports": {
        "primary": [
            r"\bmatch\b", "league", "club", r"\bteam\b", "player", "coach",
            "manager", "tournament", "championship", "fixture", "squad",
            r"\bcup\b", "athlete", "training", "stadium", "season",
            "title", "trophy", "qualifier", "final", "semi-final",
        ],
        "secondary": [
            "football", "rugby", "cricket", "tennis", "basketball", "athletics",
            "olympic", "wimbledon", "premier", "striker", "goalkeeper",
            "referee", r"\bpitch\b", "transfer", "winger", "wicket",
            "serve", "sprint", "relay", "podium", "medal", "cycling",
            "formula one", r"\bf1\b", "golf", "swimming", "boxing",
            "racing", "derby", "innings", "penalty", "offside", "try",
        ],
    },
    "business": {
        "primary": [
            "company", r"\bmarket\b", "profit", "revenue", "investment",
            r"\bbank\b", "industry", "trade", "economy", "shares",
            "shareholders", "quarterly", "earnings", "turnover", "sector",
            "firm", "corporation", "retailer", "manufacturer", "employer",
        ],
        "secondary": [
            "gdp", "inflation", "dividend", "merger", "acquisition",
            r"\bceo\b", r"\bcfo\b", "stock market", "hedge fund",
            "interest rate", "fiscal", "monetary", "startup", "venture",
            "ipo", "bankruptcy", "recession", "supply chain", "workforce",
            "redundancy", "takeover", "listed", "ftse", "dow jones",
            "interest rates", "chancellor", "budget", "deficit",
        ],
    },
    "technology": {
        "primary": [
            r"\bsoftware\b", "digital", "internet", r"\bdata\b",
            r"\bcomputer\b", r"\bnetwork\b", r"\bplatform\b", "device",
            "online", "cyber", "cloud", "tech company", "silicon valley",
            r"\bapp\b", "website", "browser", "server",
        ],
        "secondary": [
            "algorithm", "artificial intelligence", r"\bai\b",
            "machine learning", "smartphone", r"\btech\b",
            "semiconductor", "encryption", "broadband", "robotics",
            "automation", "developer", "open source", "firmware",
            "gadget", "processor", "startup", "social media",
            "streaming", "hack", "phishing", "malware",
        ],
    },
    "health": {
        "primary": [
            "hospital", "patient", r"\bhealth\b", "medical", "treatment",
            "doctor", "disease", "surgery", r"\bnhs\b", "clinical",
            "diagnosis", "medication", "pandemic", "condition",
            "healthcare", "nurse", "therapy",
        ],
        "secondary": [
            "cancer", r"\bdrug\b", "vaccine", "mental health", "epidemic",
            "pharmaceutical", r"\bgp\b", "ward", "consultant",
            "prescription", "symptom", "oncology", "antibiotic",
            "rehabilitation", "obesity", "dementia", "diabetes",
            "heart disease", "stroke", "a&e", "ambulance", "nhs trust",
        ],
    },
    "entertainment": {
        "primary": [
            "film", "movie", "music", "album", "artist", "actor",
            "actress", "director", "television", "series", "show",
            "concert", "tour", "award", "festival", "celebrity",
            "singer", "band", "drama", "comedy",
        ],
        "secondary": [
            "bafta", "oscar", "grammy", "brit award", "box office",
            "chart", "single", "debut", "sequel", "blockbuster",
            "streaming", "netflix", "bbc one", "itv", "channel 4",
            "west end", "broadway", "theatre", "exhibition",
            "gallery", "documentary", "podcast", "bbc radio",
        ],
    },
    "science": {
        "primary": [
            "scientists", "research", "study", "discovered", "species",
            "experiment", "space", "planet", "nasa", "universe",
            "fossil", "genome", "particle", "asteroid", "probe",
        ],
        "secondary": [
            "published", "journal", "findings", "laboratory", "rover",
            "telescope", "satellite", "dna", "evolution", "biology",
            "physics", "chemistry", "astronomy", "geology", "neuroscience",
            "quantum", "vaccine trial", "clinical trial", "breakthrough",
        ],
    },
    "environment": {
        "primary": [
            "climate", "environment", "wildlife", "flood", "wildfire",
            "pollution", "carbon", "species", "habitat", "conservation",
            "renewable", "emissions", "drought", "storm", "hurricane",
        ],
        "secondary": [
            "global warming", "net zero", "greenhouse", "fossil fuel",
            "solar", "wind farm", "biodiversity", "extinction", "deforestation",
            "plastic", "recycling", "ecosystem", "coral reef", "glacier",
            "sea level", "cop26", "paris agreement", "rewilding",
        ],
    },
}

_VALID_DOMAINS = list(_DOMAIN_KEYWORDS.keys()) + ["general"]
_NEEDS_REVIEW  = "__review__"   # sentinel returned when auto-tagging fails


def _matches_any(text: str, patterns: list[str]) -> bool:
    for p in patterns:
        regex = p if r"\b" in p else rf"\b{re.escape(p)}\b"
        if re.search(regex, text, re.IGNORECASE):
            return True
    return False


def _count_matches(text: str, patterns: list[str]) -> int:
    return sum(
        1 for p in patterns
        if re.search(p if r"\b" in p else rf"\b{re.escape(p)}\b", text, re.IGNORECASE)
    )


def infer_domain(document: str) -> str:
    """
    Auto-tagging rules (first match wins):
      1. 1+ primary hit       → tag immediately
      2. 2+ secondary hits    → tag
      3. Both fail            → return _NEEDS_REVIEW sentinel
    """
    text = document[:1000].lower()

    for domain, kw in _DOMAIN_KEYWORDS.items():
        if _matches_any(text, kw["primary"]):
            return domain

    for domain, kw in _DOMAIN_KEYWORDS.items():
        if _count_matches(text, kw["secondary"]) >= 2:
            return domain

    return _NEEDS_REVIEW


def annotate_pending(tasks: list[dict]) -> None:
    """
    Batch human-annotation pass for tasks that could not be auto-tagged.
    Shows total count upfront, then prompts one by one.
    Ctrl+C or blank Enter cancels remaining → tagged "general".
    """
    pending = [t for t in tasks if t["domain"] == _NEEDS_REVIEW]
    total   = len(pending)

    if not total:
        print("No tasks need human annotation.")
        return

    print(f"\n{'='*60}")
    print(f"  {total} task(s) need human annotation.")
    print(f"  Valid domains: {_VALID_DOMAINS}")
    print(f"  Leave blank + Enter (or Ctrl+C) to cancel remaining → 'general'")
    print(f"{'='*60}\n")

    for i, task in enumerate(pending, 1):
        print(f"[{i}/{total}]  {task['input']}\n")
        try:
            answer = input(f"Domain: ").strip().lower()
            if not answer:
                raise ValueError("blank")
            task["domain"] = answer if answer in _VALID_DOMAINS else "general"
        except (KeyboardInterrupt, EOFError, ValueError):
            print(f"\nCancelled — tagging remaining {total - i + 1} item(s) as 'general'.")
            for remaining in pending[i - 1:]:
                remaining["domain"] = "general"
            return
        print()


def _compression_ratio_score(document: str, summary: str) -> float:
    ratio = len(document.split()) / max(len(summary.split()), 1)
    return min(ratio / 60.0, 1.0)


def _readability_score(document: str) -> float:
    grade = textstat.flesch_kincaid_grade(document)
    return min(max(grade, 0.0) / 20.0, 1.0)


def _lexical_density_score(document: str) -> float:
    tokens = nltk.word_tokenize(document[:1000])
    if not tokens:
        return 0.0
    pos_tags = nltk.pos_tag(tokens)
    density = sum(1 for _, tag in pos_tags if tag in _CONTENT_TAGS) / len(tokens)
    return min(max((density - 0.3) / 0.4, 0.0), 1.0)


def assign_difficulty(document: str, summary: str) -> Literal["easy", "medium", "hard"]:
    """
    Combined difficulty score (weighted average of three signals):
      40% compression ratio  — how much needs to be distilled
      35% FK readability     — linguistic complexity of the source
      25% lexical density    — information density per word

    Thresholds: score < 0.35 → easy | < 0.65 → medium | else → hard
    """
    score = (
        0.40 * _compression_ratio_score(document, summary)
        + 0.35 * _readability_score(document)
        + 0.25 * _lexical_density_score(document)
    )
    if score < 0.35:
        return "easy"
    if score < 0.65:
        return "medium"
    return "hard"


RUBRIC = (
    "Score the model's summary on three dimensions (1–5 each):\n"
    "1. Faithfulness — does it contain only information from the article? "
    "Penalise hallucinations heavily.\n"
    "2. Informativeness — does it capture the single most important fact?\n"
    "3. Fluency — is it grammatically correct and natural?\n"
    "Return a JSON object: {\"faithfulness\": int, \"informativeness\": int, \"fluency\": int, \"overall\": float}"
)


def row_to_task(row: dict, split: str, index: int) -> SummarizationTask:
    return SummarizationTask(
        task_id    = f"summ_xsum_{split}_{index:06d}",
        difficulty = assign_difficulty(row["document"], row["summary"]),
        domain     = infer_domain(row["document"]),
        input      = row["document"],
        expected   = row["summary"],
        rubric     = RUBRIC,
        metadata   = {
            "source"     : DATASET_NAME,
            "original_id": row["id"],
            "split"      : split,
            "date_added" : str(date.today()),
        },
    )

## 4. Sample & convert

In [14]:
def sample_split(split_name: str, n: int | None) -> list[dict]:
    split = dataset[split_name]
    indices = list(range(len(split)))
    if n is not None:
        indices = random.sample(indices, min(n, len(indices)))
    tasks = [row_to_task(split[i], split_name, i) for i in indices]
    print(f"{split_name:10s}: {len(tasks)} tasks")
    return [t.model_dump() for t in tasks]


all_tasks = (
    sample_split("train",      N_TRAIN)
    + sample_split("validation", N_VALIDATION)
    + sample_split("test",       N_TEST)
)

print(f"\nTotal tasks: {len(all_tasks)}")

train     : 500 tasks
validation: 100 tasks
test      : 100 tasks

Total tasks: 700


In [15]:
# ── Human annotation pass ─────────────────────────────────────────────────────
# Auto-tagged tasks are untouched. Only tasks that failed both rules are shown.
annotate_pending(all_tasks)


  47 task(s) need human annotation.
  Valid domains: ['politics', 'crime', 'sports', 'business', 'technology', 'health', 'entertainment', 'science', 'environment', 'general']
  Leave blank + Enter (or Ctrl+C) to cancel remaining → 'general'

[1/47]  The fire had been smoking at Alexandra Docks since 20,000 tonnes of wood first caught alight on 5 December.
Residents had been advised to stay indoors and keep windows and doors closed after the initial blaze.
South Wales Fire and Rescue Service said the fire, which was found to be accidental, was put out on Monday.


[2/47]  Ben Emmerson was suspended in September over concerns about his conduct, but the suspension was lifted the next day when he resigned, allowing him to keep working for the inquiry for two months.
The inquiry faces claims it failed to fully investigate potential wrongdoing.
Mr Emmerson categorically denies sexual assault, bullying or other misconduct.
The inquiry denies it received any complaint of sexual assault.
It sa

## 5. Inspect difficulty & domain distribution

In [16]:
from collections import Counter

difficulty_counts = Counter(t["difficulty"] for t in all_tasks)
domain_counts     = Counter(t["domain"]     for t in all_tasks)

print("Difficulty distribution:", dict(difficulty_counts))
print("Domain distribution     :", dict(domain_counts))

Difficulty distribution: {'easy': 118, 'medium': 555, 'hard': 27}
Domain distribution     : {'health': 31, 'crime': 150, 'sports': 166, 'entertainment': 25, 'politics': 199, 'business': 50, 'general': 45, 'technology': 19, 'science': 11, 'environment': 4}


## 6. Save to JSONL

In [17]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for task in all_tasks:
        f.write(json.dumps(task, ensure_ascii=False) + "\n")

print(f"Saved {len(all_tasks)} tasks → {OUTPUT_FILE.resolve()}")

Saved 700 tasks → /home/sristi/Documents/langchain-crash-course-main/evaluation_benchmark_project/llm_evaluation_benchmarking_framework/task_bank/summarization_tasks.jsonl


## 7. Reload & validate round-trip

In [18]:
from pydantic import TypeAdapter

adapter = TypeAdapter(Task)

loaded = []
with OUTPUT_FILE.open(encoding="utf-8") as f:
    for line in f:
        loaded.append(adapter.validate_json(line))

print(f"Validated {len(loaded)} tasks — all parsed as SummarizationTask:",
      all(isinstance(t, SummarizationTask) for t in loaded))
print("\nSample task:")
print(loaded[0].model_dump_json(indent=2))

Validated 700 tasks — all parsed as SummarizationTask: True

Sample task:
{
  "task_id": "summ_xsum_train_110054",
  "difficulty": "easy",
  "domain": "health",
  "input": "City councillors said animals go through a great deal of suffering for the production of the pate.\nAnimal rights campaigners have hailed the move, but some of Sao Paulo's best-known chefs have voiced concern.\nFoie gras, originally a French delicacy, is produced worldwide.\nSeveral countries, including Britain, Germany, Italy and Argentina, have banned its production. But the sale of the pate is still allowed in most of them.\nThe Sao Paulo city council has set a fine of 5,000 reais (Â£1,000) for restaurants and bars that break the new law - which will take effect in 45 days.\n\"Foie gras is an appetiser for the wealthy,\" said the law's author, city councillor Laercio Benko.\n\"It does not benefit human health and to make it, the birds are submitted to a lot of suffering,'' said Mr Benko.\nHowever Sao Paulo-based 